## Task 1: Corpus Creation and Preprocessing

In [1]:
import re
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import nltk
import json

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/sumith/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sumith/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sumith/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
# Load data from wikepedia.txt
try:
    with open('wikepedia.txt', 'r', encoding='utf-8') as f:
        academic_corpus = f.read()
except FileNotFoundError:
    print("Error: wikepedia.txt not found in the current directory.")
    academic_corpus = ""

print(f"Original corpus length: {len(academic_corpus)} characters")
if academic_corpus:
    print(f"Number of sentences: {len(sent_tokenize(academic_corpus))}")
else:
    print("Warning: No text loaded from wikepedia.txt")

Original corpus length: 64358 characters
Number of sentences: 1067


In [3]:
# Preprocessing function
def preprocess_text(text, remove_stopwords=False):
    """
    Preprocess text: lowercase, remove punctuation, tokenize
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters but keep spaces and letters
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # Tokenize into words
    tokens = text.split()
    
    # Remove stopwords if specified
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    else:
        tokens = [token for token in tokens if len(token) > 0]
    
    return tokens

# Preprocess the corpus
print("Preprocessing corpus...")
tokens = preprocess_text(academic_corpus, remove_stopwords=False)

print(f"\nPreprocessed corpus statistics:")
print(f"Total tokens: {len(tokens)}")
print(f"Unique tokens (vocabulary size): {len(set(tokens))}")
print(f"\nFirst 50 tokens:")
print(' '.join(tokens[:50]))

Preprocessing corpus...

Preprocessed corpus statistics:
Total tokens: 7821
Unique tokens (vocabulary size): 2512

First 50 tokens:
wikipedia is a free online encyclopedia created collaboratively by volunteers around the world the project was launched on january 15 2001 and has since become one of the most visited websites on the internet wikipedia is available in over 300 languages and contains millions of articles written by millions of


In [4]:
# Save preprocessed data
preprocessed_data = {
    'tokens': tokens,
    'vocabulary': list(set(tokens)),
    'corpus_info': {
        'total_tokens': len(tokens),
        'unique_tokens': len(set(tokens)),
        'original_length': len(academic_corpus)
    }
}

print("Preprocessing complete!")
print(f"Corpus ready for model building with {len(tokens)} tokens and vocabulary size {len(set(tokens))}")

Preprocessing complete!
Corpus ready for model building with 7821 tokens and vocabulary size 2512


## Task 2: Model Building (Bigram & Trigram Language Model)

In [5]:
# Build Bigrams
def create_bigrams(tokens):
    """
    Create bigrams from token sequence
    Returns list of (word_i, word_i+1) tuples
    """
    bigrams = []
    for i in range(len(tokens) - 1):
        bigrams.append((tokens[i], tokens[i+1]))
    return bigrams

# Build Trigrams
def create_trigrams(tokens):
    """
    Create trigrams from token sequence
    Returns list of (word_i, word_i+1, word_i+2) tuples
    """
    trigrams = []
    for i in range(len(tokens) - 2):
        trigrams.append((tokens[i], tokens[i+1], tokens[i+2]))
    return trigrams

# Create n-grams
print("Creating n-grams...")
bigrams = create_bigrams(tokens)
trigrams = create_trigrams(tokens)

print(f"Total bigrams: {len(bigrams)}")
print(f"Total trigrams: {len(trigrams)}")
print(f"\nFirst 10 bigrams: {bigrams[:10]}")
print(f"\nFirst 10 trigrams: {trigrams[:10]}")

Creating n-grams...
Total bigrams: 7820
Total trigrams: 7819

First 10 bigrams: [('wikipedia', 'is'), ('is', 'a'), ('a', 'free'), ('free', 'online'), ('online', 'encyclopedia'), ('encyclopedia', 'created'), ('created', 'collaboratively'), ('collaboratively', 'by'), ('by', 'volunteers'), ('volunteers', 'around')]

First 10 trigrams: [('wikipedia', 'is', 'a'), ('is', 'a', 'free'), ('a', 'free', 'online'), ('free', 'online', 'encyclopedia'), ('online', 'encyclopedia', 'created'), ('encyclopedia', 'created', 'collaboratively'), ('created', 'collaboratively', 'by'), ('collaboratively', 'by', 'volunteers'), ('by', 'volunteers', 'around'), ('volunteers', 'around', 'the')]


In [6]:
# Calculate Bigram Conditional Probabilities
def calculate_bigram_probabilities(bigrams):
    """
    Calculate P(wi | wi-1) for all bigrams
    Returns dictionary: {word_i-1: {word_i: probability}}
    """
    # Count occurrences of each bigram
    bigram_counts = Counter(bigrams)
    
    # Count occurrences of each word in first position (context)
    context_counts = Counter([bigram[0] for bigram in bigrams])
    
    # Calculate conditional probabilities
    bigram_probs = defaultdict(dict)
    
    for (w1, w2), count in bigram_counts.items():
        # P(w2 | w1) = count(w1, w2) / count(w1)
        prob = count / context_counts[w1]
        bigram_probs[w1][w2] = {
            'probability': prob,
            'count': count
        }
    
    return dict(bigram_probs), bigram_counts, context_counts

# Calculate Trigram Conditional Probabilities
def calculate_trigram_probabilities(trigrams):
    """
    Calculate P(wi | wi-2, wi-1) for all trigrams
    Returns dictionary: {(word_i-2, word_i-1): {word_i: probability}}
    """
    # Count occurrences of each trigram
    trigram_counts = Counter(trigrams)
    
    # Count occurrences of each context (first two words)
    context_counts = Counter([(t[0], t[1]) for t in trigrams])
    
    # Calculate conditional probabilities
    trigram_probs = defaultdict(dict)
    
    for (w1, w2, w3), count in trigram_counts.items():
        context = (w1, w2)
        # P(w3 | w1, w2) = count(w1, w2, w3) / count(w1, w2)
        prob = count / context_counts[context]
        trigram_probs[context][w3] = {
            'probability': prob,
            'count': count
        }
    
    return dict(trigram_probs), trigram_counts, context_counts

# Calculate probabilities
print("Calculating bigram probabilities...")
bigram_probs, bigram_counts, bigram_context_counts = calculate_bigram_probabilities(bigrams)

print("Calculating trigram probabilities...")
trigram_probs, trigram_counts, trigram_context_counts = calculate_trigram_probabilities(trigrams)

print(f"\nBigram model: {len(bigram_probs)} unique first words")
print(f"Trigram model: {len(trigram_probs)} unique word pairs")

Calculating bigram probabilities...
Calculating trigram probabilities...

Bigram model: 2511 unique first words
Trigram model: 6848 unique word pairs


In [7]:
# Display example bigram probabilities
print("Example Bigram Probabilities (word by word):")
print("\nP(wi | 'machine')")
if 'machine' in bigram_probs:
    probs = bigram_probs['machine']
    sorted_probs = sorted(probs.items(), key=lambda x: x[1]['probability'], reverse=True)
    for word, data in sorted_probs[:5]:
        print(f"  P('{word}' | 'machine') = {data['probability']:.4f} (count: {data['count']})")
else:
    print("  'machine' not found in bigram model")

print("\nP(wi | 'learning')")
if 'learning' in bigram_probs:
    probs = bigram_probs['learning']
    sorted_probs = sorted(probs.items(), key=lambda x: x[1]['probability'], reverse=True)
    for word, data in sorted_probs[:5]:
        print(f"  P('{word}' | 'learning') = {data['probability']:.4f} (count: {data['count']})")
else:
    print("  'learning' not found in bigram model")

Example Bigram Probabilities (word by word):

P(wi | 'machine')
  P('learning' | 'machine') = 0.5000 (count: 4)
  P('translation' | 'machine') = 0.3750 (count: 3)
  P('understanding' | 'machine') = 0.1250 (count: 1)

P(wi | 'learning')
  P('uses' | 'learning') = 0.1053 (count: 2)
  P('discovers' | 'learning') = 0.1053 (count: 2)
  P('trains' | 'learning') = 0.1053 (count: 2)
  P('and' | 'learning') = 0.0526 (count: 1)
  P('natural' | 'learning') = 0.0526 (count: 1)


In [8]:
# Display example trigram probabilities
print("Example Trigram Probabilities (conditional on previous two words):")
print("\nP(wi | 'machine', 'learning')")
if ('machine', 'learning') in trigram_probs:
    probs = trigram_probs[('machine', 'learning')]
    sorted_probs = sorted(probs.items(), key=lambda x: x[1]['probability'], reverse=True)
    for word, data in sorted_probs[:5]:
        print(f"  P('{word}' | 'machine', 'learning') = {data['probability']:.4f} (count: {data['count']})")
else:
    print("  ('machine', 'learning') not found in trigram model")

print("\nP(wi | 'deep', 'learning')")
if ('deep', 'learning') in trigram_probs:
    probs = trigram_probs[('deep', 'learning')]
    sorted_probs = sorted(probs.items(), key=lambda x: x[1]['probability'], reverse=True)
    for word, data in sorted_probs[:5]:
        print(f"  P('{word}' | 'deep', 'learning') = {data['probability']:.4f} (count: {data['count']})")
else:
    print("  ('deep', 'learning') not found in trigram model")

Example Trigram Probabilities (conditional on previous two words):

P(wi | 'machine', 'learning')
  P('natural' | 'machine', 'learning') = 0.2500 (count: 1)
  P('data' | 'machine', 'learning') = 0.2500 (count: 1)
  P('enables' | 'machine', 'learning') = 0.2500 (count: 1)
  P('applications' | 'machine', 'learning') = 0.2500 (count: 1)

P(wi | 'deep', 'learning')
  P('uses' | 'deep', 'learning') = 0.2500 (count: 1)
  P('has' | 'deep', 'learning') = 0.2500 (count: 1)
  P('models' | 'deep', 'learning') = 0.2500 (count: 1)
  P('translation' | 'deep', 'learning') = 0.2500 (count: 1)


## Task 3: Next Word Prediction Function

In [9]:
def predict_next_words(input_text, bigram_probs, trigram_probs, top_k=5, use_trigram=True):
    """
    Predict the next likely words given user input.
    
    Parameters:
    - input_text: user's typed input (string)
    - bigram_probs: bigram probability model
    - trigram_probs: trigram probability model
    - top_k: number of suggestions to return
    - use_trigram: if True, use trigram model; else use bigram
    
    Returns:
    - List of (word, probability) tuples sorted by probability
    """
    # Preprocess input
    input_tokens = preprocess_text(input_text, remove_stopwords=False)
    
    if not input_tokens:
        return []
    
    suggestions = []
    
    if use_trigram and len(input_tokens) >= 2:
        # Use trigram: P(wi | wi-2, wi-1)
        context = (input_tokens[-2], input_tokens[-1])
        
        if context in trigram_probs:
            probs = trigram_probs[context]
            suggestions = [(word, data['probability']) for word, data in probs.items()]
            suggestions.sort(key=lambda x: x[1], reverse=True)
            suggestions = suggestions[:top_k]
        else:
            # Fall back to bigram if trigram context not found
            last_word = input_tokens[-1]
            if last_word in bigram_probs:
                probs = bigram_probs[last_word]
                suggestions = [(word, data['probability']) for word, data in probs.items()]
                suggestions.sort(key=lambda x: x[1], reverse=True)
                suggestions = suggestions[:top_k]
    else:
        # Use bigram: P(wi | wi-1)
        last_word = input_tokens[-1]
        
        if last_word in bigram_probs:
            probs = bigram_probs[last_word]
            suggestions = [(word, data['probability']) for word, data in probs.items()]
            suggestions.sort(key=lambda x: x[1], reverse=True)
            suggestions = suggestions[:top_k]
    
    return suggestions

print("Next word prediction function created!")

Next word prediction function created!


In [10]:
# Test the prediction function with various inputs
test_inputs = [
    "machine learning",
    "deep learning",
    "neural network",
    "the model",
    "training a",
    "gradient descent",
    "convolutional neural",
    "supervised learning"
]

print("="*70)
print("NEXT WORD PREDICTION - TEST RESULTS")
print("="*70)

for input_text in test_inputs:
    print(f"\nInput: '{input_text}'")
    print("-" * 70)
    
    # Trigram predictions
    trigram_suggestions = predict_next_words(input_text, bigram_probs, trigram_probs, top_k=5, use_trigram=True)
    
    if trigram_suggestions:
        print("Top 5 Trigram-based Suggestions:")
        for i, (word, prob) in enumerate(trigram_suggestions, 1):
            print(f"  {i}. '{word}' - Probability: {prob:.4f}")
    else:
        print("No trigram suggestions found, falling back to bigram...")
        bigram_suggestions = predict_next_words(input_text, bigram_probs, trigram_probs, top_k=5, use_trigram=False)
        if bigram_suggestions:
            print("Top 5 Bigram-based Suggestions:")
            for i, (word, prob) in enumerate(bigram_suggestions, 1):
                print(f"  {i}. '{word}' - Probability: {prob:.4f}")

NEXT WORD PREDICTION - TEST RESULTS

Input: 'machine learning'
----------------------------------------------------------------------
Top 5 Trigram-based Suggestions:
  1. 'natural' - Probability: 0.2500
  2. 'data' - Probability: 0.2500
  3. 'enables' - Probability: 0.2500
  4. 'applications' - Probability: 0.2500

Input: 'deep learning'
----------------------------------------------------------------------
Top 5 Trigram-based Suggestions:
  1. 'uses' - Probability: 0.2500
  2. 'has' - Probability: 0.2500
  3. 'models' - Probability: 0.2500
  4. 'translation' - Probability: 0.2500

Input: 'neural network'
----------------------------------------------------------------------
Top 5 Trigram-based Suggestions:
  1. 'analysis' - Probability: 0.6667
  2. 'monitoring' - Probability: 0.3333

Input: 'the model'
----------------------------------------------------------------------
Top 5 Trigram-based Suggestions:
  1. 'performance' - Probability: 0.4286
  2. 'configuration' - Probability: 0.1

## Task 4: Create Interactive Streamlit Application

In [11]:
# Save the models to files for use in Streamlit app
import pickle

model_data = {
    'bigram_probs': bigram_probs,
    'trigram_probs': trigram_probs,
    'bigram_counts': dict(bigram_counts),
    'trigram_counts': dict(trigram_counts),
    'vocabulary': set(tokens)
}

# Save as pickle
with open('ngram_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model data saved to 'ngram_model.pkl'")
print(f"Model contains:")
print(f"  - {len(bigram_probs)} bigram contexts")
print(f"  - {len(trigram_probs)} trigram contexts")
print(f"  - {len(set(tokens))} unique tokens in vocabulary")

Model data saved to 'ngram_model.pkl'
Model contains:
  - 2511 bigram contexts
  - 6848 trigram contexts
  - 2512 unique tokens in vocabulary


In [12]:
# Generate comprehensive test results and statistics
test_results = []

for input_text in test_inputs:
    suggestions = predict_next_words(input_text, bigram_probs, trigram_probs, top_k=5, use_trigram=True)
    test_results.append({
        'input': input_text,
        'predictions': suggestions
    })

print("Test results generated and ready for report.")

Test results generated and ready for report.


## Summary Statistics

In [13]:
# Print comprehensive summary
print("\n" + "="*70)
print("LANGUAGE MODEL SUMMARY STATISTICS")
print("="*70)

print("\n1. CORPUS STATISTICS")
print("-" * 70)
print(f"   Original text length: {len(academic_corpus)} characters")
print(f"   Total tokens after preprocessing: {len(tokens)}")
print(f"   Unique tokens (vocabulary size): {len(set(tokens))}")
print(f"   Number of sentences: {len(sent_tokenize(academic_corpus))}")
print(f"   Average tokens per sentence: {len(tokens) / len(sent_tokenize(academic_corpus)):.2f}")

print("\n2. N-GRAM STATISTICS")
print("-" * 70)
print(f"   Total bigrams: {len(bigrams)}")
print(f"   Unique bigrams: {len(bigram_counts)}")
print(f"   Total trigrams: {len(trigrams)}")
print(f"   Unique trigrams: {len(trigram_counts)}")

print("\n3. BIGRAM MODEL")
print("-" * 70)
print(f"   Unique first words (contexts): {len(bigram_probs)}")
print(f"   Average next words per context: {sum(len(v) for v in bigram_probs.values()) / len(bigram_probs):.2f}")

print("\n4. TRIGRAM MODEL")
print("-" * 70)
print(f"   Unique word pair contexts: {len(trigram_probs)}")
print(f"   Average next words per context: {sum(len(v) for v in trigram_probs.values()) / len(trigram_probs):.2f}")

print("\n5. MOST COMMON WORDS")
print("-" * 70)
word_freq = Counter(tokens)
most_common = word_freq.most_common(10)
for i, (word, count) in enumerate(most_common, 1):
    print(f"   {i}. '{word}': {count} times ({count/len(tokens)*100:.2f}%)")


LANGUAGE MODEL SUMMARY STATISTICS

1. CORPUS STATISTICS
----------------------------------------------------------------------
   Original text length: 64358 characters
   Total tokens after preprocessing: 7821
   Unique tokens (vocabulary size): 2512
   Number of sentences: 1067
   Average tokens per sentence: 7.33

2. N-GRAM STATISTICS
----------------------------------------------------------------------
   Total bigrams: 7820
   Unique bigrams: 6849
   Total trigrams: 7819
   Unique trigrams: 7637

3. BIGRAM MODEL
----------------------------------------------------------------------
   Unique first words (contexts): 2511
   Average next words per context: 2.73

4. TRIGRAM MODEL
----------------------------------------------------------------------
   Unique word pair contexts: 6848
   Average next words per context: 1.12

5. MOST COMMON WORDS
----------------------------------------------------------------------
   1. 'and': 493 times (6.30%)
   2. 'to': 111 times (1.42%)
   3. '